In [ ]:
import tensorflow as tf
import numpy as np

from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Dropout 
from tensorflow.keras.layers import Conv2DTranspose
from tensorflow.keras.layers import concatenate

#from test_utils import summary, comparator

In [ ]:
import os
import shutil
import random

# Define paths
image_folder = "/Users/kakumanisravya/Desktop/project/CameraRGB/"
mask_folder = "/Users/kakumanisravya/Desktop/project/CameraMask/"
train_folder = "/Users/kakumanisravya/Desktop/project/train/"
test_folder = "/Users/kakumanisravya/Desktop/project/test/"

# Create directories if they don't exist
os.makedirs(os.path.join(train_folder, "Images"), exist_ok=True)
os.makedirs(os.path.join(train_folder, "Masks"), exist_ok=True)
os.makedirs(os.path.join(test_folder, "Images"), exist_ok=True)
os.makedirs(os.path.join(test_folder, "Masks"), exist_ok=True)



image_filenames = [filename for filename in os.listdir(image_folder) if os.path.isfile(os.path.join(image_folder, filename))]
mask_filenames = [filename for filename in os.listdir(mask_folder) if os.path.isfile(os.path.join(mask_folder, filename))]

# Define proportions for train and test sets
train_split = 0.8
test_split = 0.2

# Calculate number of images for each set
num_images = len(image_filenames)
num_train = int(train_split * num_images)
num_test = num_images - num_train

# Split filenames into train and test sets
train_image_filenames = image_filenames[:num_train]
test_image_filenames = image_filenames[num_train:]

train_mask_filenames = mask_filenames[:num_train]
test_mask_filenames = mask_filenames[num_train:]

# Copy images and masks to train folder
for img_filename, mask_filename in zip(train_image_filenames, train_mask_filenames):
    shutil.copy(os.path.join(image_folder, img_filename), os.path.join(train_folder, "Images", img_filename))
    shutil.copy(os.path.join(mask_folder, mask_filename), os.path.join(train_folder, "Masks", mask_filename))

# Copy images and masks to test folder
for img_filename, mask_filename in zip(test_image_filenames, test_mask_filenames):
    shutil.copy(os.path.join(image_folder, img_filename), os.path.join(test_folder, "Images", img_filename))
    shutil.copy(os.path.join(mask_folder, mask_filename), os.path.join(test_folder, "Masks", mask_filename))
    
    
train_image_filenames = [train_folder+"Images/"+i for i in train_image_filenames]
test_image_filenames = [test_folder+"Images/"+i for i in test_image_filenames]

train_mask_filenames = [train_folder+"Masks/"+i for i in train_mask_filenames]
test_mask_filenames = [test_folder+"Masks/"+i for i in test_mask_filenames]

In [ ]:
train_dataset=tf.data.Dataset.from_tensor_slices((train_image_filenames,train_mask_filenames))
test_dataset=tf.data.Dataset.from_tensor_slices((test_image_filenames,test_mask_filenames))

In [ ]:
def process_path(image_path, mask_path):
    img = tf.io.read_file(image_path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.convert_image_dtype(img, tf.float32)

    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_png(mask, channels=3)
    mask = tf.math.reduce_max(mask, axis=-1, keepdims=True)
    return img, mask

def preprocess(image, mask):
    input_image = tf.image.resize(image, (96, 128), method='nearest')
    input_mask = tf.image.resize(mask, (96, 128), method='nearest')

    input_image = input_image / 255.

    return input_image, input_mask

train_image_ds = train_dataset.map(process_path)
train_processed_image_ds = train_image_ds.map(preprocess)
test_image_ds = test_dataset.map(process_path)
test_processed_image_ds = test_image_ds.map(preprocess)

In [ ]:
#defining a convblock method which has 2 conv block a variable dropout and maxpooling 
def conv_block(inputs=None, n_filters=32, dropout_prob=0, max_pooling=True):
    conv = Conv2D(n_filters,3,activation='relu',padding='same',kernel_initializer='he_normal')(inputs)
    conv = Conv2D(n_filters,3,activation='relu',padding='same',kernel_initializer='he_normal')(conv)
   
    if dropout_prob > 0:
        conv = Dropout(rate=dropout_prob)(conv)
    if max_pooling:
        next_layer = MaxPooling2D(2,2)(conv)
    else:
        next_layer = conv
    skip_connection = conv
    
    return next_layer, skip_connection

In [ ]:
"""
        upsampling block
        expansive_input -- Input tensor from previous layer
        contractive_input -- Input tensor from previous skip layer
        conv -- Tensor output
    """
def upsampling_block(expansive_input, contractive_input, n_filters=32):
    up = Conv2DTranspose(n_filters,3,strides=2,padding='same')(expansive_input)
    # Merge the previous output and the skiplayer 
    merge = concatenate([up, contractive_input], axis=3)
    conv = Conv2D(n_filters,3,activation='relu',padding='same',kernel_initializer='he_normal')(merge)
    conv = Conv2D(n_filters,3,activation='relu',padding='same',kernel_initializer='he_normal')(conv)
    return conv

In [ ]:
def unet_model(input_size=(96, 128, 3), n_filters=32, n_classes=23):
    inputs = Input(input_size)

    #Contracting path (encoding)
    cblock1 = conv_block(inputs, n_filters)
    # output of each block to be the input of the next block.
    # Double the number of filters at each new step
    cblock2 = conv_block(cblock1[0], n_filters*2)
    cblock3 = conv_block(cblock2[0], n_filters*4)
    cblock4 = conv_block(cblock3[0], n_filters*8, dropout_prob=0.3)
    cblock5 = conv_block(cblock4[0],  n_filters*16, dropout_prob=0.3, max_pooling=False)

    # Expanding Path (decoding)
    # Use the cblock5[0] is expansive_input and cblock4[1] is contractive_input
    ublock6 = upsampling_block(cblock5[0], cblock4[1],  n_filters * 8)
    #half the number of filters of the previous block
    ublock7 = upsampling_block(ublock6, cblock3[1], n_filters * 4)
    ublock8 = upsampling_block(ublock7, cblock2[1], n_filters * 2)
    ublock9 = upsampling_block(ublock8, cblock1[1], n_filters)

    conv9 = Conv2D(n_filters,3,activation='relu',padding='same',kernel_initializer='he_normal')(ublock9)
    conv10 = Conv2D(n_classes, 1, padding='same')(conv9)
    model = tf.keras.Model(inputs=inputs, outputs=conv10)

    return model

In [ ]:
unet = unet_model((96,128,3),32,23)
unet.summary()

In [ ]:
unet.compile(optimizer='adam',loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),metrics=['accuracy'])

In [ ]:

train_dataset = train_processed_image_ds.cache().batch(32)
print(train_processed_image_ds.element_spec)
model_history = unet.fit(train_dataset, epochs=40)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(model_history.history["accuracy"])

In [ ]:
test_dataset = test_processed_image_ds.cache().batch(32)
test_loss, test_accuracy = unet.evaluate(test_dataset)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

In [ ]:
def display(display_list):
    plt.figure(figsize=(15, 15))

    title = ['Input Image', 'True Mask', 'Predicted Mask']

    for i in range(len(display_list)):
        plt.subplot(1, len(display_list), i+1)
        plt.title(title[i])
        plt.imshow(tf.keras.preprocessing.image.array_to_img(display_list[i]))
        plt.axis('off')
    plt.show()

In [ ]:
def create_mask(pred_mask):
    pred_mask = tf.argmax(pred_mask, axis=-1)
    pred_mask = pred_mask[..., tf.newaxis]
    return pred_mask[0]

In [ ]:
def show_predictions(dataset, num):
    for image, mask in dataset.take(num):
        pred_mask = unet.predict(image)
        display([image[0], mask[0], create_mask(pred_mask)])

In [ ]:
show_predictions(test_dataset, 6)

In [ ]:
def fcn_model(input_size, n_filters, n_classes):
    inputs = Input(shape=input_size)
    # Contracting path
    c1 = Conv2D(n_filters, (3, 3), activation='relu', padding='same')(inputs)
    c1 = Conv2D(n_filters, (3, 3), activation='relu', padding='same')(c1)
    c1 = Dropout(0.3)(c1)  # Add dropout
    p1 = MaxPooling2D((2, 2))(c1)

    c2 = Conv2D(n_filters * 2, (3, 3), activation='relu', padding='same')(p1)
    c2 = Conv2D(n_filters * 2, (3, 3), activation='relu', padding='same')(c2)
    c2 = Dropout(0.3)(c2)  # Add dropout
    p2 = MaxPooling2D((2, 2))(c2)

    c3 = Conv2D(n_filters * 4, (3, 3), activation='relu', padding='same')(p2)
    c3 = Conv2D(n_filters * 4, (3, 3), activation='relu', padding='same')(c3)
    c3 = Dropout(0.3)(c3)  # Add dropout
    p3 = MaxPooling2D((2, 2))(c3)

    c4 = Conv2D(n_filters * 8, (3, 3), activation='relu', padding='same')(p3)
    c4 = Conv2D(n_filters * 8, (3, 3), activation='relu', padding='same')(c4)
    c4 = Dropout(0.3)(c4)  # Add dropout
    p4 = MaxPooling2D((2, 2))(c4)

    # Expanding Path
    u1 = Conv2DTranspose(n_filters * 8, (3, 3), strides=2, padding='same')(p4)
    c5 = Conv2D(n_filters * 8, (3, 3), activation='relu', padding='same')(u1)

    u2 = Conv2DTranspose(n_filters * 4, (3, 3), strides=2, padding='same')(c5)
    c6 = Conv2D(n_filters * 4, (3, 3), activation='relu', padding='same')(u2)

    u3 = Conv2DTranspose(n_filters * 2, (3, 3), strides=2, padding='same')(c6)
    c7 = Conv2D(n_filters * 2, (3, 3), activation='relu', padding='same')(u3)

    u4 = Conv2DTranspose(n_filters, (3, 3), strides=2, padding='same')(c7)
    c8 = Conv2D(n_filters, (3, 3), activation='relu', padding='same')(u4)

    # Output Layer
    outputs = Conv2D(n_classes, (1, 1), padding='same')(c8)

    
    return tf.keras.Model(inputs=inputs, outputs=outputs)


In [ ]:
fcn = fcn_model((96, 128, 3), 32, 23)
fcn.summary()

In [ ]:
fcn.compile(optimizer='adam', loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=['accuracy'])

In [ ]:
train_dataset = train_processed_image_ds.cache().shuffle(500).batch(32)
fcn_model_history = fcn.fit(train_dataset, epochs=40)

In [ ]:
plt.plot(fcn_model_history.history["accuracy"])

In [ ]:
test_dataset = test_processed_image_ds.cache().batch(32)
test_loss, test_accuracy = fcn.evaluate(test_dataset)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

In [ ]:
def show_fcn_predictions(dataset, num):
    for image, mask in dataset.take(num):
            pred_mask = fcn.predict(image)
            display([image[0], mask[0], create_mask(pred_mask)])

In [ ]:
show_fcn_predictions(train_dataset, 6)